In [ ]:
# CVS aircraft detector training - runs unchanged on Kaggle and Colab. Edit the settings, then run.
#  Kaggle: attach two datasets: one with train.py + common.py, one with cvs_dataset (folder or .zip). GPU: T4/P100.
#  Colab : put train.py, common.py and cvs_dataset.zip (or the folder) in Google Drive at MyDrive/cvs. Runtime: GPU.
MODEL  = "yolo26s.pt"     # "yolov8s.pt" | "yolo26s.pt"
AUG    = "degrade"        # "none" | "degrade"
EPOCHS = 100
BATCH  = 32               # 32 fits 16 GB for s models
FROM   = None             # fine-tune from weights, e.g. "/kaggle/input/cvs-weights/best.pt"
EXTRA  = []               # e.g. ["--limit", "64"] for a smoke test

import glob, os, shutil, subprocess, sys, zipfile
ON_KAGGLE = os.path.isdir("/kaggle/input")
if not ON_KAGGLE:
    from google.colab import drive
    drive.mount("/content/drive")
IN   = "/kaggle/input" if ON_KAGGLE else "/content/drive/MyDrive/cvs"
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.219", "albumentations==2.0.8"], check=True)

def find(pattern, root):
    hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
    return hits[0] if hits else None

code = os.path.join(WORK, "cvs")
os.makedirs(code, exist_ok=True)
for f in ("train.py", "common.py"):
    shutil.copy(find(f, IN) or sys.exit(f"{f} not found under {IN}"), code)

data = find("data.yaml", IN)
if data is None:                                   # dataset uploaded as a zip that was not auto-extracted
    z = find("*.zip", IN) or sys.exit(f"no data.yaml or zip under {IN}")
    zipfile.ZipFile(z).extractall(os.path.join(WORK, "data"))
    data = find("data.yaml", os.path.join(WORK, "data"))
print("code:", code, "| data:", data)

name = f"{os.path.splitext(os.path.basename(MODEL))[0]}_{AUG}" + ("_ft" if FROM else "")
cmd = [sys.executable, "train.py", "--model", MODEL, "--aug", AUG, "--epochs", str(EPOCHS), "--batch", str(BATCH),
       "--data", data, "--name", name, "--workers", "4"] + (["--from", FROM] if FROM else []) + EXTRA
p = subprocess.Popen(cmd, cwd=code, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                     encoding="utf-8", errors="replace", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
for line in p.stdout:
    print(line, end="")
if p.wait():
    raise SystemExit(f"train.py failed with exit code {p.returncode}")

run_dir = os.path.join(code, "runs", name)
archive = shutil.make_archive(os.path.join(WORK, name), "zip", run_dir)
if not ON_KAGGLE:
    shutil.copy(archive, IN)
print("weights:", os.path.join(run_dir, "weights", "best.pt"), "| download:", archive)
